<a href="https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/translations/ru/notebooks/ch07_deep_rl_ru.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); 
padding: 40px; border-radius: 12px; color: white; margin-bottom: 20px;">
<h1 style="font-size:2.2em; margin:0 0 10px 0;">Глава 7. Аппроксимация функции ценности и DQN</h1>
<p style="font-size:1.1em; opacity:0.85; margin:0;">
От табличного Q-Learning к Deep Q-Networks — experience replay, target-сети и аблейшен-исследование на CartPole-v1.
</p>
</div>

### Цели обучения
- Понять, зачем нужны нейросети для больших пространств состояний
- Реализовать 3-слойный MLP DQN на PyTorch
- Освоить experience replay и target network
- Провести аблейшен-эксперименты, чтобы увидеть вклад каждого компонента
- Сравнить с базовой линией линейного Q-Learning

## Подготовка окружения

> **Colab / Kaggle**: запустите ячейку ниже один раз. Используется только CPU — обучение занимает ~5–7 минут на бесплатном Colab.

In [ ]:
# ~30 s install
!pip install -q "gymnasium[classic-control]" torch matplotlib numpy tqdm


In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
import math
import matplotlib
import matplotlib.pyplot as plt
from collections import deque
from tqdm.auto import trange, tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


<div style="background:#1b4332;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">1 · Линейная базовая линия Q-Learning</h2></div>

Прежде чем переходить к глубоким сетям, установим базовую линию с **линейным аппроксиматором**.
Признаки состояния — сырые наблюдения (4-мерный вектор для CartPole). Q-значения — линейная функция этих признаков: $Q(s,a;\mathbf{w}) = \mathbf{w}_a^\top \phi(s)$.

Это нижняя граница для сравнения — если DQN не обходит линейную модель, что-то не так.

In [ ]:
class LinearQNetwork(nn.Module):
    """Linear Q-function approximator (no hidden layers)."""
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc = nn.Linear(state_dim, action_dim)

    def forward(self, x):
        return self.fc(x)


def train_linear_q(n_episodes=300, lr=1e-2, gamma=0.99,
                   eps_start=1.0, eps_end=0.05, eps_decay=0.995,
                   seed=SEED):
    """Train a linear Q-network on CartPole-v1 (no replay, no target net)."""
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)
    state_dim = env.observation_space.shape[0]   # 4
    action_dim = env.action_space.n               # 2

    net = LinearQNetwork(state_dim, action_dim).to(DEVICE)
    optimizer = optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    episode_rewards = []
    eps = eps_start

    for ep in trange(n_episodes, desc="LinearQ", leave=False):
        state, _ = env.reset()
        total_reward = 0
        done = False

        while not done:
            s_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)

            # Epsilon-greedy action selection
            if random.random() < eps:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = net(s_t).argmax(dim=1).item()

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward

            # Online 1-step TD update
            s_next = torch.tensor(next_state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                target = reward + (0.0 if done else gamma * net(s_next).max().item())
            target_t = torch.tensor([[target]], dtype=torch.float32, device=DEVICE)
            pred = net(s_t)[:, action].unsqueeze(1)
            loss = loss_fn(pred, target_t)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            state = next_state

        eps = max(eps_end, eps * eps_decay)
        episode_rewards.append(total_reward)

    env.close()
    return episode_rewards


linear_rewards = train_linear_q(n_episodes=300)
print(f"Linear Q  — last-50-ep mean: {np.mean(linear_rewards[-50:]):.1f}")


<div style="background:#1d3557;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">2 · Глубокая Q-сеть (DQN)</h2></div>

DQN (Mnih и др., 2015) вводит два ключевых трюка стабилизации:

| Компонент | Назначение |
|-----------|----------|
| **Experience Replay** | Разрывает временные корреляции; повышает эффективность по выборке |
| **Target Network** | Фиксирует TD-цель на `C` шагов; уменьшает осцилляции |

Минимизируется функция потерь:

$$
\mathcal{L}(\theta) = \mathbb{E}_{(s,a,r,s')\sim\mathcal{D}}\left[\left(r + \gamma \max_{a'} Q(s',a';\theta^-) - Q(s,a;\theta)\right)^2\right]
$$

где $\theta^-$ — замороженные параметры target-сети.

In [ ]:
# ── Replay Buffer ────────────────────────────────────────────────────────────
class ReplayBuffer:
    def __init__(self, capacity=10_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (
            torch.tensor(np.array(s), dtype=torch.float32, device=DEVICE),
            torch.tensor(a, dtype=torch.long, device=DEVICE),
            torch.tensor(r, dtype=torch.float32, device=DEVICE),
            torch.tensor(np.array(ns), dtype=torch.float32, device=DEVICE),
            torch.tensor(d, dtype=torch.float32, device=DEVICE),
        )

    def __len__(self):
        return len(self.buffer)


# ── Q-Network (3-layer MLP) ───────────────────────────────────────────────────
class DQNNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )

    def forward(self, x):
        return self.net(x)


# ── DQN Trainer ──────────────────────────────────────────────────────────────
def train_dqn(
    n_episodes=300,
    lr=1e-3,
    gamma=0.99,
    batch_size=64,
    buffer_size=10_000,
    target_update_freq=10,   # episodes between target-net syncs
    eps_start=1.0,
    eps_end=0.01,
    eps_decay=200,           # exponential decay steps
    use_replay=True,
    use_target=True,
    seed=SEED,
    desc="DQN",
):
    """
    Full DQN.  Toggle use_replay / use_target for ablation.
    eps_decay is the half-life in steps for exponential schedule.
    """
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    online_net = DQNNetwork(state_dim, action_dim).to(DEVICE)
    target_net = DQNNetwork(state_dim, action_dim).to(DEVICE)
    target_net.load_state_dict(online_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(online_net.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    replay    = ReplayBuffer(buffer_size) if use_replay else None

    episode_rewards = []
    total_steps = 0

    for ep in trange(n_episodes, desc=desc, leave=False):
        state, _ = env.reset()
        total_reward = 0
        done = False

        while not done:
            # Exponential epsilon schedule
            eps = eps_end + (eps_start - eps_end) * math.exp(-total_steps / (eps_decay * 10))
            total_steps += 1

            s_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            if random.random() < eps:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = online_net(s_t).argmax(dim=1).item()

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward

            if use_replay:
                replay.push(state, action, reward, next_state, float(done))
                if len(replay) >= batch_size:
                    s, a, r, ns, d = replay.sample(batch_size)
                    _update(online_net, target_net if use_target else online_net,
                            optimizer, loss_fn, s, a, r, ns, d, gamma)
            else:
                # Online TD update (no replay)
                s_np  = np.array([state],      dtype=np.float32)
                ns_np = np.array([next_state], dtype=np.float32)
                s_b  = torch.tensor(s_np,  device=DEVICE)
                ns_b = torch.tensor(ns_np, device=DEVICE)
                a_b  = torch.tensor([action], device=DEVICE)
                r_b  = torch.tensor([reward], dtype=torch.float32, device=DEVICE)
                d_b  = torch.tensor([float(done)], device=DEVICE)
                _update(online_net,
                        target_net if use_target else online_net,
                        optimizer, loss_fn, s_b, a_b, r_b, ns_b, d_b, gamma)
            state = next_state

        # Sync target network every target_update_freq episodes
        if use_target and (ep % target_update_freq == 0):
            target_net.load_state_dict(online_net.state_dict())

        episode_rewards.append(total_reward)

    env.close()
    return episode_rewards


def _update(online, target, optimizer, loss_fn, s, a, r, ns, d, gamma):
    with torch.no_grad():
        max_next_q = target(ns).max(dim=1).values
        td_target   = r + gamma * max_next_q * (1 - d)
    q_vals = online(s).gather(1, a.unsqueeze(1)).squeeze(1)
    loss   = loss_fn(q_vals, td_target)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(online.parameters(), 10.0)
    optimizer.step()


In [ ]:
# ~3-5 min total on Colab CPU
print("Training full DQN...")
dqn_rewards = train_dqn(n_episodes=300, desc="Full DQN")

print("Training DQN without experience replay...")
no_replay_rewards = train_dqn(n_episodes=300, use_replay=False, desc="No Replay")

print("Training DQN without target network...")
no_target_rewards = train_dqn(n_episodes=300, use_target=False, desc="No Target")

print(f"Full DQN     — last-50 mean: {np.mean(dqn_rewards[-50:]):.1f}")
print(f"No Replay    — last-50 mean: {np.mean(no_replay_rewards[-50:]):.1f}")
print(f"No Target    — last-50 mean: {np.mean(no_target_rewards[-50:]):.1f}")
print(f"Linear Q     — last-50 mean: {np.mean(linear_rewards[-50:]):.1f}")


<div style="background:#4a1942;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">3 · Сравнение кривых обучения</h2></div>

Для наглядности тренда сглаживаем вознаграждения скользящим окном.

In [ ]:
def smooth(x, window=20):
    return np.convolve(x, np.ones(window)/window, mode="valid")


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: all four curves ────────────────────────────────────────────────────
ax = axes[0]
curves = {
    "Full DQN":          (dqn_rewards,        "#2ecc71"),
    "DQN – No Replay":   (no_replay_rewards,  "#e74c3c"),
    "DQN – No Target":   (no_target_rewards,  "#f39c12"),
    "Linear Q-learning": (linear_rewards,     "#3498db"),
}
for label, (rewards, color) in curves.items():
    sm = smooth(rewards)
    ax.plot(sm, label=label, color=color, linewidth=2)

ax.set_xlabel("Episode", fontsize=12)
ax.set_ylabel("Total Reward (smoothed, w=20)", fontsize=12)
ax.set_title("DQN Ablation Study — CartPole-v1", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.axhline(475, color="gray", linestyle="--", alpha=0.5, label="near-optimal")
ax.set_ylim(0, 510)
ax.grid(alpha=0.3)

# ── Right: bar chart of final performance ────────────────────────────────────
ax2 = axes[1]
labels = ["Full DQN", "No Replay", "No Target", "Linear Q"]
means  = [np.mean(r[-50:]) for r in [dqn_rewards, no_replay_rewards, no_target_rewards, linear_rewards]]
colors = ["#2ecc71", "#e74c3c", "#f39c12", "#3498db"]
bars   = ax2.bar(labels, means, color=colors, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f"{val:.0f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax2.set_ylabel("Mean Reward (last 50 episodes)", fontsize=12)
ax2.set_title("Final Performance Comparison", fontsize=13, fontweight="bold")
ax2.set_ylim(0, 530)
ax2.axhline(475, color="gray", linestyle="--", alpha=0.5)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("ch07_ablation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved: ch07_ablation.png")


<div style="background:#1b4332;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">4 · Затухание $\varepsilon$ и диагностика Q-значений</h2></div>

Понимание расписания исследования и масштаба Q-значений помогает диагностировать обучение.

In [ ]:
# ── Epsilon schedule visualisation ──────────────────────────────────────────
steps = np.arange(0, 60_000)
eps_curve = 0.01 + (1.0 - 0.01) * np.exp(-steps / 2000)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(steps, eps_curve, color="#e74c3c", linewidth=2)
ax.fill_between(steps, eps_curve, alpha=0.2, color="#e74c3c")
ax.set_xlabel("Training Step")
ax.set_ylabel("Epsilon (exploration probability)")
ax.set_title("Exponential Epsilon-Greedy Schedule")
ax.axhline(0.01, color="gray", linestyle="--", label="eps_min = 0.01")
ax.legend()
ax.grid(alpha=0.3)

# ── Rolling variance of reward ───────────────────────────────────────────────
ax2 = axes[1]
window = 30
for label, rewards, color in [
        ("Full DQN", dqn_rewards, "#2ecc71"),
        ("No Target", no_target_rewards, "#f39c12")]:
    arr = np.array(rewards)
    roll_std = [arr[max(0,i-window):i+1].std() for i in range(len(arr))]
    ax2.plot(roll_std, label=label, color=color, linewidth=1.5, alpha=0.85)

ax2.set_xlabel("Episode")
ax2.set_ylabel("Rolling Std of Reward (w=30)")
ax2.set_title("Training Stability: Full DQN vs No-Target DQN")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("ch07_diagnostics.png", dpi=120, bbox_inches="tight")
plt.show()


<div style="background:#1d3557;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">5 · Архитектура сети и сводка</h2></div>

In [ ]:
# Print architecture summary
sample_net = DQNNetwork(state_dim=4, action_dim=2)
print("DQN Architecture (CartPole-v1)")
print("=" * 40)
print(sample_net)
total_params = sum(p.numel() for p in sample_net.parameters())
print(f"\nTotal parameters: {total_params:,}")

lin_net = LinearQNetwork(state_dim=4, action_dim=2)
lin_params = sum(p.numel() for p in lin_net.parameters())
print(f"Linear Q params : {lin_params:,}")

# Summary table
fig, ax = plt.subplots(figsize=(9, 3))
ax.axis("off")
table_data = [
    ["Method",           "Replay", "Target Net", "Params", "Mean Reward (last 50)"],
    ["Full DQN",         "Yes",    "Yes",         f"{total_params:,}", f"{np.mean(dqn_rewards[-50:]):.0f}"],
    ["DQN – No Replay",  "No",     "Yes",         f"{total_params:,}", f"{np.mean(no_replay_rewards[-50:]):.0f}"],
    ["DQN – No Target",  "Yes",    "No",          f"{total_params:,}", f"{np.mean(no_target_rewards[-50:]):.0f}"],
    ["Linear Q",         "No",     "No",          f"{lin_params:,}",   f"{np.mean(linear_rewards[-50:]):.0f}"],
]
tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0],
               cellLoc="center", loc="center",
               colColours=["#2c3e50"]*5)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.3, 1.8)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_text_props(color="white", fontweight="bold")
plt.title("Chapter 7 — Results Summary", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("ch07_summary_table.png", dpi=120, bbox_inches="tight")
plt.show()


<div style="background: #0d2137; padding: 20px; border-radius: 8px; border: 1px solid #1f6feb; margin-top: 20px;">
  <h3 style="color: #79c0ff; margin-top: 0;">Глава 7 — ключевые выводы</h3>
  <ul style="color: #c9d1d9; line-height: 1.8;">
    <li><strong>Experience Replay</strong>: разрывает временную корреляцию обучающих данных; критичен для стабильности на длинных задачах.</li>
    <li><strong>Target Network</strong>: предотвращает <<погоню за движущейся целью>>; особенно полезна в начале обучения.</li>
    <li><strong>Глубина важна</strong>: 2–3 скрытых слоя превосходят линейные базовые линии на нелинейных ландшафтах ценности.</li>
    <li><strong>Затухание $\varepsilon$</strong>: исследование должно убывать достаточно медленно, чтобы сеть успела обучиться до перехода к эксплуатации.</li>
  </ul>
</div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #f0a500; margin-top: 20px;">
  <strong style="color: #f0a500;">Дальше:</strong> в главе 8 переходим от value-based к policy-based методам — REINFORCE и далее.
</div>